# Baseline A — Language-specific descriptive (fastText CC-300)

Computes pairwise cosine distance matrices and clustering structure for each
(language, domain) combination using language-specific fastText CC-300 vectors.
This is the human-readable descriptive baseline for the ph-project.

**Sub-issue:** `ph-project-ssa.4`

**Output artifacts:**
- `results/baseline_A/dendrograms/<lang>_<domain>.png` — 9 dendrogram figures
- `results/baseline_A/silhouette/<lang>_<domain>.png` — 9 silhouette sweep figures
- `results/baseline_A/summary.csv` — summary table (9 rows)

## Setup

In [1]:
# Ensure the repo root (parent of notebooks/) is on sys.path so that
# `import baselines` resolves correctly regardless of how the kernel was
# launched or what the CWD is.
import sys
import pathlib

_NOTEBOOK_DIR = pathlib.Path(__file__).resolve().parent if '__file__' in dir() else pathlib.Path('.').resolve()
_REPO_ROOT = _NOTEBOOK_DIR.parent if _NOTEBOOK_DIR.name == 'notebooks' else _NOTEBOOK_DIR

if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

print(f'Repo root: {_REPO_ROOT}')
print(f'sys.path[0]: {sys.path[0]}')

Repo root: /home/anna/ph-project-ssa-4-baseline-A
sys.path[0]: /home/anna/ph-project-ssa-4-baseline-A


In [2]:
import warnings

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.cluster.hierarchy import dendrogram

from baselines import SUPPORTED_LANGUAGES
from baselines.vectors import load_fasttext
from baselines.distances import extract_term_vectors, cosine_distance_matrix
from baselines.clustering import agglomerative_linkage, kmeans_silhouette_sweep

# Suppress gensim/numpy deprecation noise during loading
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

REPO_ROOT = _REPO_ROOT
CANON_ROOT = REPO_ROOT / 'canon-terms'
RESULTS_ROOT = REPO_ROOT / 'results' / 'baseline_A'

(RESULTS_ROOT / 'dendrograms').mkdir(parents=True, exist_ok=True)
(RESULTS_ROOT / 'silhouette').mkdir(parents=True, exist_ok=True)

# Domains — hardcoded (documented in canon-terms/SCHEMA.md)
DOMAINS = ['color', 'emotion', 'kinship']

# Languages from the single source of truth in baselines/__init__.py
LANGUAGES = sorted(SUPPORTED_LANGUAGES)  # deterministic order: ['en', 'es', 'ru']

# k_range cap: k can only go up to N-1 where N is the number of found terms.
# For each (lang, domain), we compute k_range = range(2, min(K_MAX, n_found-1) + 1).
K_MAX = 8  # upper bound on k (inclusive)

print(f'Languages: {LANGUAGES}')
print(f'Domains:   {DOMAINS}')
print(f'Results:   {RESULTS_ROOT}')

Languages: ['en', 'es', 'ru']
Domains:   ['color', 'emotion', 'kinship']
Results:   /home/anna/ph-project-ssa-4-baseline-A/results/baseline_A


## Step 1 — Load canon terms

Read all 9 (language, domain) YAML files and collect term lists.
Each file is parsed according to `canon-terms/SCHEMA.md`.

In [3]:
def load_canon_terms(lang: str, domain: str) -> list:
    """Read canon term strings from canon-terms/<lang>/<domain>.yaml."""
    path = CANON_ROOT / lang / f'{domain}.yaml'
    with open(path) as f:
        data = yaml.safe_load(f)
    return [entry['term'] for entry in data['terms']]


# Pre-load all term lists
terms_by_lang_domain = {}
for lang in LANGUAGES:
    for domain in DOMAINS:
        terms = load_canon_terms(lang, domain)
        terms_by_lang_domain[(lang, domain)] = terms
        print(f'{lang}/{domain}: {len(terms)} terms')

en/color: 11 terms
en/emotion: 18 terms
en/kinship: 27 terms
es/color: 11 terms
es/emotion: 22 terms
es/kinship: 32 terms
ru/color: 12 terms
ru/emotion: 19 terms
ru/kinship: 34 terms


## Step 2 — Load fastText CC-300 vectors

Load each language's CC-300 model **once** and reuse it across domains.
Each `.bin` file is ~7 GB; loading all three uses ~9 GB RAM (host has 64 GB).

In [4]:
vectors_by_lang = {}
for lang in LANGUAGES:
    print(f'Loading fastText CC-300 for lang={lang!r}...')
    vectors_by_lang[lang] = load_fasttext(lang, 'cc')
    print(f'  dim={vectors_by_lang[lang].vector_size}, loaded.')

print('\nAll vectors loaded.')

Loading fastText CC-300 for lang='en'...


  dim=300, loaded.
Loading fastText CC-300 for lang='es'...


  dim=300, loaded.
Loading fastText CC-300 for lang='ru'...


  dim=300, loaded.

All vectors loaded.


## Step 3 — Extract term vector matrices

For each (language, domain), extract a `(n_found, 300)` matrix using
`baselines.distances.extract_term_vectors` with `strategy='head'` and
the per-language head-position rule (EN/RU: right-headed; ES: left-headed).

FastText CC-300 supports subword composition, so OOV is not expected.

In [5]:
matrices = {}
found_terms = {}
oov_rates = {}

for lang in LANGUAGES:
    kv = vectors_by_lang[lang]
    for domain in DOMAINS:
        terms = terms_by_lang_domain[(lang, domain)]
        matrix, mask = extract_term_vectors(
            terms, kv, strategy='head', lang=lang, domain=domain
        )
        ft = [t for t, m in zip(terms, mask) if m]
        oov = 1.0 - mask.sum() / len(terms) if terms else 0.0
        matrices[(lang, domain)] = matrix
        found_terms[(lang, domain)] = ft
        oov_rates[(lang, domain)] = oov
        status = '' if mask.all() else f'  [OOV: {int((~mask).sum())} terms]'
        print(f'{lang}/{domain}: {matrix.shape[0]}/{len(terms)} found{status}')

print('\nExtraction complete.')

en/color: 11/11 found
en/emotion: 18/18 found
en/kinship: 27/27 found
es/color: 11/11 found
es/emotion: 22/22 found
es/kinship: 32/32 found
ru/color: 12/12 found
ru/emotion: 19/19 found
ru/kinship: 34/34 found

Extraction complete.


## Step 4 — Cosine distance matrices

Compute the pairwise cosine distance matrix for each (language, domain)
using `baselines.distances.cosine_distance_matrix` which wraps
`scipy.spatial.distance.pdist(..., metric='cosine') + squareform`.

In [6]:
distance_matrices = {}

for lang in LANGUAGES:
    for domain in DOMAINS:
        X = matrices[(lang, domain)]
        D = cosine_distance_matrix(X)
        distance_matrices[(lang, domain)] = D
        print(f'{lang}/{domain}: D shape={D.shape}, range=[{D.min():.4f}, {D.max():.4f}]')

print('\nDistance matrices computed.')

en/color: D shape=(11, 11), range=[0.0000, 0.4983]
en/emotion: D shape=(18, 18), range=[0.0000, 0.8245]
en/kinship: D shape=(27, 27), range=[0.0000, 0.5951]
es/color: D shape=(11, 11), range=[0.0000, 0.5071]
es/emotion: D shape=(22, 22), range=[0.0000, 1.0033]
es/kinship: D shape=(32, 32), range=[0.0000, 0.8513]
ru/color: D shape=(12, 12), range=[0.0000, 0.4009]
ru/emotion: D shape=(19, 19), range=[0.0000, 0.8991]
ru/kinship: D shape=(34, 34), range=[0.0000, 0.7324]

Distance matrices computed.


## Step 5 — Agglomerative clustering + dendrograms

For each (language, domain):
1. Compute a hierarchical linkage matrix using `agglomerative_linkage` (UPGMA, `method='average'`).
2. Plot the dendrogram with matplotlib, labelling leaves with canon term strings.
3. Save to `results/baseline_A/dendrograms/<lang>_<domain>.png`.

In [7]:
linkage_matrices = {}

for lang in LANGUAGES:
    for domain in DOMAINS:
        D = distance_matrices[(lang, domain)]
        ft = found_terms[(lang, domain)]
        n = len(ft)

        Z = agglomerative_linkage(D, method='average')
        linkage_matrices[(lang, domain)] = Z

        fig, ax = plt.subplots(figsize=(max(8, n * 0.6), 5))
        dendrogram(
            Z,
            labels=ft,
            ax=ax,
            leaf_rotation=45,
            leaf_font_size=9,
        )
        ax.set_title(
            f'Dendrogram — {lang.upper()} / {domain}\n'
            f'(fastText CC-300, average linkage, cosine distance)'
        )
        ax.set_ylabel('Cosine distance')
        ax.set_xlabel('Canon terms')
        fig.tight_layout()

        out_path = RESULTS_ROOT / 'dendrograms' / f'{lang}_{domain}.png'
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'Saved: {out_path.name}')

print('\nAll dendrograms saved.')

Saved: en_color.png
Saved: en_emotion.png


Saved: en_kinship.png
Saved: es_color.png


Saved: es_emotion.png
Saved: es_kinship.png


Saved: ru_color.png
Saved: ru_emotion.png


Saved: ru_kinship.png

All dendrograms saved.


## Step 6 — k-means silhouette sweep

For each (language, domain), run k-means for `k = 2 .. min(K_MAX, n_found-1)`
and compute the silhouette score (`metric='cosine'`).

**k_range cap note:** silhouette requires at least 2 clusters and at most
`n_found - 1` clusters. If a domain has fewer than `K_MAX + 1` terms,
we cap `k_range` accordingly.

Each silhouette plot is saved to `results/baseline_A/silhouette/<lang>_<domain>.png`.

In [8]:
silhouette_scores = {}
best_k = {}

for lang in LANGUAGES:
    for domain in DOMAINS:
        X = matrices[(lang, domain)]
        n = X.shape[0]

        # Cap k_range: k must be at least 2 and at most n-1
        k_max_for_domain = min(K_MAX, n - 1)
        if k_max_for_domain < 2:
            print(f'{lang}/{domain}: only {n} terms — cannot compute silhouette (need >= 3). Skipping.')
            silhouette_scores[(lang, domain)] = {}
            best_k[(lang, domain)] = -1
            continue

        k_range = range(2, k_max_for_domain + 1)
        scores = kmeans_silhouette_sweep(X, k_range)
        silhouette_scores[(lang, domain)] = scores

        bk = max(scores, key=scores.__getitem__)
        best_k[(lang, domain)] = bk

        # Plot
        fig, ax = plt.subplots(figsize=(6, 4))
        ks = sorted(scores.keys())
        vals = [scores[k] for k in ks]
        ax.plot(ks, vals, marker='o', linewidth=1.5)
        ax.axvline(bk, linestyle='--', color='red', alpha=0.6,
                   label=f'best k={bk} ({scores[bk]:.3f})')
        ax.set_title(f'Silhouette sweep — {lang.upper()} / {domain}\n'
                     f'(k-means, metric=cosine)')
        ax.set_xlabel('k (number of clusters)')
        ax.set_ylabel('Silhouette score')
        ax.set_xticks(ks)
        ax.legend()
        fig.tight_layout()

        out_path = RESULTS_ROOT / 'silhouette' / f'{lang}_{domain}.png'
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'{lang}/{domain}: best k={bk} ({scores[bk]:.4f}) — saved {out_path.name}')

print('\nAll silhouette plots saved.')

en/color: best k=8 (0.0721) — saved en_color.png


en/emotion: best k=3 (-0.0284) — saved en_emotion.png
en/kinship: best k=2 (-0.0525) — saved en_kinship.png


es/color: best k=6 (0.0520) — saved es_color.png


es/emotion: best k=2 (0.0074) — saved es_emotion.png
es/kinship: best k=2 (0.0301) — saved es_kinship.png


ru/color: best k=4 (0.0979) — saved ru_color.png


ru/emotion: best k=2 (0.1037) — saved ru_emotion.png
ru/kinship: best k=2 (0.1771) — saved ru_kinship.png

All silhouette plots saved.


## Step 7 — Summary table

Assemble a CSV with one row per (language, domain) covering:
- `n_terms` — total terms in the canon list
- `n_found` — terms found in the fastText vocab (should be n_terms for CC-300)
- `oov_rate` — fraction of terms missing from vocab
- `best_k` — k with highest silhouette score
- `best_silhouette` — silhouette score at best_k

Saved to `results/baseline_A/summary.csv`.

In [9]:
rows = []
for lang in LANGUAGES:
    for domain in DOMAINS:
        terms = terms_by_lang_domain[(lang, domain)]
        n_terms = len(terms)
        n_found = matrices[(lang, domain)].shape[0]
        oov_rate = oov_rates[(lang, domain)]
        bk = best_k[(lang, domain)]
        scores = silhouette_scores[(lang, domain)]
        best_sil = scores[bk] if bk != -1 and scores else float('nan')
        rows.append({
            'language': lang,
            'domain': domain,
            'n_terms': n_terms,
            'n_found': n_found,
            'oov_rate': round(oov_rate, 4),
            'best_k': bk,
            'best_silhouette': round(float(best_sil), 4) if not np.isnan(best_sil) else float('nan'),
        })

df = pd.DataFrame(rows)
csv_path = RESULTS_ROOT / 'summary.csv'
df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')
df

Saved: /home/anna/ph-project-ssa-4-baseline-A/results/baseline_A/summary.csv


,language,domain,n_terms,n_found,oov_rate,best_k,best_silhouette
0,en,color,11,11,0.0,8,0.0721
1,en,emotion,18,18,0.0,3,-0.0284
2,en,kinship,27,27,0.0,2,-0.0525
3,es,color,11,11,0.0,6,0.0520
4,es,emotion,22,22,0.0,2,0.0074
5,es,kinship,32,32,0.0,2,0.0301
6,ru,color,12,12,0.0,4,0.0979
7,ru,emotion,19,19,0.0,2,0.1037
8,ru,kinship,34,34,0.0,2,0.1771


## Step 8 — Discussion

### What this baseline measures

Sub-baseline A gives us the *descriptive* view of each language's semantic
domain structure using static fastText CC-300 embeddings. The dendrograms show
which canon terms cluster together under average-linkage hierarchical
clustering; the silhouette sweep suggests the number of distinct groupings
the embedding space supports for each (language, domain).

### What to look for

**Color:** Russian color should show more fine-grained structure than English
or Spanish because Russian lexicalises the blue/light-blue distinction as two
basic color terms (синий / голубой). If this distinction is encoded in the
CC-300 embedding space, the Russian color dendrogram should show синий and
голубой in a separate cluster from the other color terms.

**Emotion:** Cross-linguistic divergence in emotion semantics is expected to
be larger than for color (pre-registered prediction P2). The silhouette
structure for Russian emotion should capture culture-specific clustering —
e.g., тоска and тревога may cluster distinctly from the Ekman basic emotions.

**Kinship:** Russian and Spanish both have richer lexicalized kinship systems
than English. The dendrograms may show more fine-grained structure for RU/ES.

### Limitations

This is a *static embedding* baseline — it captures distributional co-occurrence
patterns, not the attentional structures hypothesised to encode cultural
specificity. Sub-baseline B (topological, ssa.5) passes these same distance
matrices through persistent homology to characterise their *shape*; the main
experiment (Phase 3) uses mBERT attention graphs instead.

In [10]:
# Verify all expected artifacts exist
expected_dendrograms = [RESULTS_ROOT / 'dendrograms' / f'{l}_{d}.png'
                        for l in LANGUAGES for d in DOMAINS]
expected_silhouette = [RESULTS_ROOT / 'silhouette' / f'{l}_{d}.png'
                       for l in LANGUAGES for d in DOMAINS]

missing = []
for p in expected_dendrograms + expected_silhouette + [csv_path]:
    if not p.exists():
        missing.append(str(p))

if missing:
    print(f'MISSING ARTIFACTS ({len(missing)}):')
    for m in missing:
        print(f'  {m}')
else:
    print(f'All artifacts present:')
    print(f'  {len(expected_dendrograms)} dendrograms')
    print(f'  {len(expected_silhouette)} silhouette plots')
    print(f'  summary.csv')

assert not missing, f'Missing artifacts: {missing}'

All artifacts present:
  9 dendrograms
  9 silhouette plots
  summary.csv
